# 当前正式知识 pipeline

原始资料 → 清洗与概念身份 → 正文筛选、Qwen/Gemma图片筛选 → `joint_paragraphs` → `final_review` → 程序校验与保存。

`joint_paragraphs`只提取有依据的图文知识；`final_review`一次检查各组提取稿并写最终版本。原有verify/repair/relationship/merge多轮链不再执行。模型只用简单资料号和图号，来源与图片信息由程序附回。


**执行前请重启内核。** 禁止在已跑过旧代码的内核里逐个热替换类继续执行。代码、提示词或配置改变后使用新的RUN；相同版本才复用checkpoint。当前修改尚未重跑端到端，旧run原样保留。

### 当前正式执行链
原始datasets → 文档清洗/过滤/去重、图片字节检查 → 概念身份及文字相关性 → **Qwen3.8图片初筛 → Gemma31B独立复核入选图 → 分歧暂缓** → 原生图文关联与相似度补充 → 32K容量组装 → v5图文段落提炼 → 引用/像素核验 → 按需一次跨批整合 → 知识文件。

新RUN默认重新执行新图片筛选。`image_identity_definitions`可按concept_ref明确范围；未配置时使用原概念记录中的名称、别名、QID、简介等，剔除上游图片判定。补充limits为空不再误挡完整判断，但缺字段和协议矛盾仍暂缓。细物种身份仍需来源证据，双模型一致不等于已核验。

8001已有Gemma31时复用；默认`image_review_service='borrow'`自动借用并恢复本地GPU服务；`external`只使用外部已准备服务。这里只查看当前RUN的结果。

### 主线算子：作用、输入、输出

|算子|输入|输出与作用|
|---|---|---|
|read_records → SelectSourceRecords → Concept/Document/ImageFromRecord|原始采集清单|按同一概念选择规则先过滤再转换，保留来源行号与源字段；QID页面映射仍完整保留，避免掩盖歧义|
|SelectConcept 与 join|概念过滤条件、三张表|选中概念及关联材料；不限制原始扫描条数|
|ReadDocument → CleanDocument → FilterDocumentBlocks|文档行、保存的原始页面|清洗正文块、原文位置、分离的图注和参考资料|
|CheckImage|图片行和原始文件|文件可用性、字节校验、尺寸等；缺图单独记录|
|PrepareIdentity → identity → ApplyIdentity|概念、材料预览|接受/排除材料及身份歧义；预览范围明确保留|
|BuildSourceBlocks → relevance → ApplyBlockSelection|身份接受的正文块|按概念相关性保留原文块，不改写原文|
|SelectAvailableImages → Qwen初筛 → Gemma独立复核 → ApplyConfirmedImageSelection|实际图片、概念名及身份定义（不传图片caption/预标注/上游判断）|相关可用图、排除依据及未确定范围|
|PrepareRoutingMaterials|相关正文、图片、页面块中原生图片引用|材料表和可恢复的原生图文关联|
|EmbedParagraphBatch / EncodeImageTextMaterials|正文、实际图片|Qwen 文本向量、SigLIP2 图文向量，仅服务材料组织|
|RouteByTokenBudget → BuildRoutedJointRequest|向量、原生关联、容量配置|容量内的联合请求；没有匹配正文的图仍有独立处理机会|
|joint_paragraphs → ApplyParagraphs|原文、实际像素|主题段落、模型选择的相关互补图片、逐字文档引用或图片区域引用（image_refs）；无最终图片张数上限|
|verify_paragraphs → ApplyParagraphReview|提炼结果、同一原文与像素|逐块及整个标题/正文/图片组合的核验；移除上游接受理由，只看原始证据|
|PlanCrossBatchReview → BatchRelationshipReviews → review_relationships|同批与跨批段落及向量/引文|重复、互补、条件差异、冲突等候选关系；无候选则旁路|
|BuildLocalMergeGroups → merge_paragraphs → verify_merged_paragraphs|相关候选、原文、真实图片|局部整合内容和重新核验；超容量任务显式暂缓|
|ApplyLocalIntegration|原段落、局部整合结果|替换实际消费的块；内容移走的旧主题标记待重查|
|PrepareTopicRepairs → repair_topics → ApplyTopicRepairs|删文后失效的标题/正文/图片、原始证据|按需修复并再次看图核验；失败内部保存，不发布空标题纯图主题|
|RetainReviewedTopics → FormatTopicArticle|有效主题与来源表|三部分主题文章，发布阶段不按分数筛图|
|FinalKnowledgeRecord → checkpoint|概念、文档、图片、主题文章、过程审计|每概念一行的 knowledge_base.jsonl|

提示词源码在 `ops/prompts/`：`identity.yaml`、`relevance.yaml`、`select_images.yaml`、`joint_paragraphs.yaml`、`verify_paragraphs.yaml`、`review_relationships.yaml`、`merge_paragraphs.yaml`、`verify_merged_paragraphs.yaml`、`repair_topics.yaml`。state 内只保存冻结副本。

`quality_pipeline.py`中的`verify_topics`、`repair_topics`是原生Dataset链的组合函数，业务处理仍由上述op/prompt执行。实际试验仍有残留重复及文章割裂，不能因接通了算子就称质量合格。

默认只做一次按需整合；无候选即零关系调用。`global_material_audit=True`可显式开启全库未关联审计，开启时禁用入口下推。清洗不等于事实正确，首次核验与改写后的核验保留。


## 1．处理与查看配置
MAX_RECORDS默认None：每个指定文件不限制扫描条数；填写整数才截断。IDS、材料分批、每次模型材料预算和调用预算是独立配置，保持显式。默认view_saved只查看已有最终文件，不启动全量扫描。


In [ ]:
from pathlib import Path
import sys, asyncio, itertools, random, json
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
# 不对运行中的Dataset热替换算子：发现旧内核时停止，重启后再读取checkpoint。
def _assert_current_kernel():
    import os
    try:
        shell = get_ipython()
    except NameError:
        return  # CLI是独立Python进程。
    if shell is None or not hasattr(shell, 'kernel'):
        return
    boot = next(int(line.split()[1]) for line in Path('/proc/stat').read_text().splitlines() if line.startswith('btime '))
    started = boot + int(Path('/proc/self/stat').read_text().rsplit(')', 1)[1].split()[19]) / os.sysconf('SC_CLK_TCK')
    changed = []
    for name, module in tuple(sys.modules.items()):
        if name.startswith(('curation.', 'demiflow.')):
            filename = getattr(module, '__file__', None)
            if filename and Path(filename).is_file() and Path(filename).stat().st_mtime > started:
                changed.append(name)
    if changed:
        raise RuntimeError('当前内核启动后算子代码已更新，请重启内核并重新执行初始化；禁止新旧算子混用。涉及：' + ', '.join(changed[:6]))
_assert_current_kernel()
from curation.v4.ops.filter_document_blocks import FilterDocumentBlocks
from curation.v4.ops.select_source_records import SelectSourceRecords
from curation.v4.ops.cross_batch import BatchRelationshipReviews, ApplyRelationshipReviews
from curation.v4.ops.multimodal import (SelectAvailableImages, BatchImageSelection, ApplyImageSelection,
    merge_image_decisions, SelectRelatedMaterials, BatchJointMaterials, ApplyJointExtraction,
    ApplyJointVerification, merge_joint_batches, merge_joint_scope, PrepareJointMerge, ApplyJointMerge)
from demiflow.standalone import local_data
from curation.v4.ops.material_routing import PrepareRoutingMaterials, RawPassageRows, EncodeImageTextMaterials, BuildRoutedJointRequest
from curation.v4.ops.paragraph_similarity import EmbedParagraphBatch, ParagraphRows
from curation.v4.ops.paragraphs import ApplyParagraphs, ApplyParagraphReview, SelectRetainedParagraphs
from curation.v4.ops.cross_batch import PlanCrossBatchReview, ApplyCrossBatchReview
from curation.v4.ops.paragraph_merge import ApplyParagraphMerge
from curation.v4.ops.topic_articles import TopicRows
from curation.v4.ops.token_routing import RouteByTokenBudget
from curation.v4.ops.paragraph_pipeline import (RouteConceptMaterials, PrepareVerifiedParagraphs, BuildLocalMergeGroups,
    ApplyLocalIntegration, SourceCatalog, FormatTopicArticle, FinalKnowledgeRecord)
# 业务算子：只处理数据，不隐藏上/下游Dataset。
from curation.v4.ops.dataset_operators import (
    ConceptFromRecord, DocumentFromRecord, ImageFromRecord, SelectConcept, MaterialLinks, ReadDocument, CleanDocument, CheckImage,
    CountMaterial, NestMaterial, merge_concept, merge_document, distinct,
    fill_material_counts, model_input)
from curation.v4.ops.prompt_operators import (
    SelectPassagesAndImages, BuildCandidateRecords,
    PrepareIdentity, ApplyIdentity, PrepareExtraction, ApplyExtraction,
    PrepareConsolidation, ApplyConsolidation, PrepareImageSupport, ApplyImageSupport)
from curation.v4.ops.fidelity import PrepareFidelity, ApplyFidelity
from curation.v4.ops.source_blocks import (BuildSourceBlocks, BatchSourceBlocks, ApplyBlockSelection,
    merge_block_decisions, BuildVerbatimCandidates, BatchSourceComparisons,
    ApplySourceComparison, merge_source_comparisons, ApplyComparedCandidates)
from curation.v4.ops.prompt_config import knowledge_prompt_pack, prompt_execution_options, save_prompt_config
# 下面仅为原文件发现、文件读取、版本冻结和提示词配置，不是流程对象。
from curation.v4.contracts import digest, run_lock, snapshot
from curation.v4.notebook_io import freeze_run, read_saved
from curation.v4.pipeline import DEFAULT
from curation.v4.ops.image_filter import (IMAGE_FILTER_DEFAULTS, RecordPrimaryImageSelection, PrepareImageReview, ApplyConfirmedImageSelection)
from curation.v4.image_filter_runtime import image_prompt_data, review_needed, save_image_filter_policy, validate_material_reuse
from curation.v4.local_review_service import image_review_service
from curation.v4.final_results import BuildKnowledgeRecord

# view_saved只读已有结果；execute才执行下面唯一的pipeline入口。
# None 从原始数据开始；指定父run须匹配当前图片筛选policy。
REUSE_MATERIALS = None  # 旧单模型入选材料不能跳过新筛选
MODE = 'view_saved'
RUN = ROOT / 'state/curation/v4/bench200_sample5_article_v1'
DATASET = ROOT / 'datasets/demiwtg'
# 以下参数仅用于新建run。全文件处理：IDS=None、SAMPLE_RATE=1、MAX_RECORDS=None。
# 取消这些工程预算不等于跨批整合、吞吐及质量已经验收。
IDS = ['legacy:瓶式台球', 'legacy:高原', 'legacy:高锰酸钾', 'legacy:OK手势', 'legacy:白花芍药']  # 种子20260917随机3例 + 定向回归2例；RUN/bench200_selection.json保存清单
SOURCE_SCOPE = 'collected'  # 三个原始采集文件；all再包含QID与Wiki文件
SAMPLE_RATE, SEED, MAX_RECORDS, GROUP_SIZE = 1.0, 42, None, 256
THROUGH = 'export'  # 可改identity/organize/extract/consolidate/fidelity/evidence/export
# 既有回归概念的明确身份范围；新概念使用概念记录，不按名称猜物种。
IMAGE_IDENTITY_DEFINITIONS = {'legacy:OK手势': '拇指和食指相触成环、其余手指伸展或放松的手势；相关图解和实际使用场景也可保留。', 'legacy:玻璃棒': '实验室中用于搅拌、引流等的实心玻璃棒；同属实验器材不自动属于目标。', 'legacy:白花芍药': '植物学物种 Paeonia sterniana；泛指白色芍药花或其他栽培品种不自动认证为这一物种。身份不确定时保留不确定性。'}
MODEL_CONFIG = {**IMAGE_FILTER_DEFAULTS, 'image_identity_definitions': IMAGE_IDENTITY_DEFINITIONS, 'article_mode':True,'joint_thinking':True,'joint_reasoning_effort':'low','final_review_thinking':True,'final_review_effort':'low', 'final_input_tokens':131072, 'joint_input_tokens':32768, 'text_mode':'multimodal', 'body_only':True, 'max_calls':None, 'max_output_tokens':16384,
                'temperature':0, 'timeout_s':900, 'block_unit_chars':1800,
                'block_batch_chars':8000, 'comparison_group_chars':16000, 'image_batch_size':4,
                'text_embedding_model':str(ROOT.parent/'models/Qwen3-Embedding-0.6B'),
                'image_embedding_model':str(ROOT.parent/'models/siglip2-base-patch16-224')}  # 运行知识阶段时冻结本地模型配置与预算
# 修改代码/配置后执行须使用新的RUN目录，旧结果不可覆盖。

# 查看配置只影响展示，不影响处理或模型调用；新run实际sink为RUN/'knowledge_base.jsonl'。
FINAL_FILE = RUN / 'knowledge_base.jsonl'
# view_saved展示已完成的正式v1；上方RUN仍为下一次执行的v2目录。
SAVED_RUN = ROOT / 'state/curation/v4/bench200_sample5_dual_v1'
VIEW = dict(concepts=None, limit=None, images=True)  # 只影响展示，不限制最终入选图片数量

from curation.v4.ops.article import (PrepareArticleInput, ApplyArticle, PrepareFinalReview, PublishArticle, ArticleTokenBudget, PrepareSelectionScope)


## 2．精简后的完整 pipeline

1. 读取原清单，按概念先过滤，再转换和关联文档/图片。
2. 清洗、去重、修复正文；检查图片文件。
3. 检查身份，筛选相关正文和图片。
4. 保留原生图文关联，按语义与容量组装材料。
5. 联合提炼，核对引用和实际图片。
6. 有候选才批量判断关系，最多一轮局部整合；只有改写部分重新核验。失效主题仍按需修复。
7. 保存图文知识及原始材料/过程审计。

默认不做全库未关联材料审计，不再固定第二轮整合，不再运行只记录残留关系的末轮模型检查。首次核验保留：清洗不保证模型改写或图片身份正确。

可在 MODEL_CONFIG 设置 `global_material_audit=True` 开启完整审计（会关闭入口下推）；`relationship_batch_pairs=8`、`relationship_batch_chars=24000` 控制关系请求容量。超容量不截断、不自动通过。

本轮来自bench200的200概念：固定种子20260917随机抽取瓶式台球、高原、高锰酸钾；另加OK手势、白花芍药作定向回归。完整清单及抽样依据在RUN/bench200_selection.json，最终仅展示当前运行结果。


In [ ]:
def run_pipeline(run, dataset, *, ids=None, sample_rate=1., seed=42,
                 max_records_per_source=None, group_size=32, through='gather',
                 model_config=None, project=ROOT, source_scope="all", reuse_preprocessing=None, reuse_materials=None, reuse_filter_inputs=None):
    """Dataset编排直接在这里；业务算子只处理行，不决定上下游。"""
    _assert_current_kernel()
    if through not in ['gather','identity','organize','extract','consolidate','fidelity','evidence','export']:
        raise ValueError('Unknown stopping stage')
    run, dataset = Path(run), Path(dataset)
    if source_scope not in {"all", "collected"}: raise ValueError("invalid source_scope")
    config = {**DEFAULT, **IMAGE_FILTER_DEFAULTS, **(model_config or {})}
    # 全库未关联资料审计是可选旁路；开启时关闭入口下推以保证审计完整。
    global_audit = config.get('global_material_audit', False)
    if config.get('image_annotations_file'):
        config['image_annotations_sha256'] = digest(Path(config['image_annotations_file']).read_bytes())
    settings = dict(ids=ids, sample_rate=sample_rate, seed=seed,
                    max_records_per_source=max_records_per_source, group_size=group_size,
                    model_config=config, source_scope=source_scope)
    if reuse_preprocessing is not None:settings['reuse_preprocessing']=str(Path(reuse_preprocessing).resolve())
    if reuse_materials is not None:
        validate_material_reuse(reuse_materials, config)
        if through not in {'extract','consolidate','fidelity','evidence','export'} or config.get('text_mode')!='multimodal':
            raise ValueError('Material reuse begins at multimodal joint extraction')
        if reuse_preprocessing is not None:raise ValueError('Choose one reuse boundary')
        from curation.v4.notebook_io import frozen_material_inputs, import_frozen_materials
        settings['reuse_materials']=frozen_material_inputs(reuse_materials)
    if reuse_filter_inputs is not None:
        if reuse_materials is not None or reuse_preprocessing is not None:raise ValueError('Choose one checkpoint boundary')
        if through in {'gather','identity'}:raise ValueError('Filter reuse starts after identity')
        from curation.v4.notebook_io import frozen_filter_inputs, import_frozen_materials
        settings['reuse_filter_inputs']=frozen_filter_inputs(reuse_filter_inputs,ids)
    # 只管并发锁与版本冻结，不隐藏任何业务调度；through可向后续跑。
    with run_lock(run):
        tables = run/'datasets'
        data = local_data()

        # 1. 文件快照仅供版本冻结和定位校验，不负责读取或调度。
        legacy_concepts_source = {'kind':'legacy_concepts', **snapshot(dataset/'meta/concepts.json')}
        qid_concepts_source = {'kind':'qid_concepts', **snapshot(dataset/'meta/qid_concepts.fat.jsonl.gz')}
        collected_documents_source = {'kind':'legacy_docs', **snapshot(dataset/'meta/docs.jsonl')}
        collected_images_source = {'kind':'legacy_images', **snapshot(dataset/'meta/images.jsonl')}
        wiki_pages_source = {'kind':'wiki_pages', **snapshot(dataset/'corpus/pages-en-part1.jsonl.gz')}
        active_sources=[legacy_concepts_source,collected_documents_source,collected_images_source]
        if source_scope=='all':active_sources += [qid_concepts_source,wiki_pages_source]
        version = freeze_run(run, dataset, active_sources, settings, run_pipeline, project)
        if reuse_preprocessing is not None:
            from curation.v4.notebook_io import reuse_preprocessing as import_preprocessing
            import_preprocessing(reuse_preprocessing,run,version,settings)

        if reuse_materials is not None:
            import_frozen_materials(settings['reuse_materials'],run,version)
        elif reuse_filter_inputs is not None:
            import_frozen_materials(settings['reuse_filter_inputs'],run,version)
        else:
            # 原生读取 → 保存原始解码行（含错误）→ 有效对象 → 业务字段转换。
            # read_records负责gzip/JSON解析、行号与扫描报告；map不读文件。
            # 解码失败或非对象行保留在read_*检查点，不静默丢弃原文。
            legacy_concepts_records = (data.read_records(dataset/'meta/concepts.json', format='json', item_prefix='concepts.item',
                max_records=max_records_per_source, missing='empty', report_path=run/'source_status/legacy_concepts.json')
                .filter(SelectSourceRecords('legacy_concepts', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_legacy_concepts.jsonl', version=version))
            legacy_concepts = (legacy_concepts_records
                .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                .map(ConceptFromRecord(legacy_concepts_source))
                .checkpoint(tables/'input_legacy_concepts.jsonl', version=version))

            if source_scope=='all':
                qid_concepts_records = (data.read_records(dataset/'meta/qid_concepts.fat.jsonl.gz',
                    max_records=max_records_per_source, missing='empty', report_path=run/'source_status/qid_concepts.json')
                    .filter(SelectSourceRecords('qid_concepts', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_qid_concepts.jsonl', version=version))
                qid_concepts = (qid_concepts_records
                    .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                    .map(ConceptFromRecord(qid_concepts_source))
                    .checkpoint(tables/'input_qid_concepts.jsonl', version=version))

            else:
                qid_concepts = data.from_iter(lambda: iter(()))

            collected_documents_records = (data.read_records(dataset/'meta/docs.jsonl',
                max_records=max_records_per_source, missing='empty', report_path=run/'source_status/collected_documents.json')
                .filter(SelectSourceRecords('legacy_docs', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_collected_documents.jsonl', version=version))
            collected_documents = (collected_documents_records
                .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                .map(DocumentFromRecord(collected_documents_source))
                .checkpoint(tables/'input_collected_documents.jsonl', version=version))

            collected_images_records = (data.read_records(dataset/'meta/images.jsonl',
                max_records=max_records_per_source, missing='empty', report_path=run/'source_status/collected_images.json')
                .filter(SelectSourceRecords('legacy_images', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_collected_images.jsonl', version=version))
            collected_images = (collected_images_records
                .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                .map(ImageFromRecord(collected_images_source))
                .checkpoint(tables/'input_collected_images.jsonl', version=version))

            if source_scope=='all':
                wiki_pages_records = (data.read_records(dataset/'corpus/pages-en-part1.jsonl.gz',
                    max_records=max_records_per_source, missing='empty', report_path=run/'source_status/wiki_pages.json')
                    .checkpoint(tables/'read_wiki_pages.jsonl', version=version))
                wiki_pages = (wiki_pages_records
                    .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                    .map(DocumentFromRecord(wiki_pages_source))
                    .checkpoint(tables/'input_wiki_pages.jsonl', version=version))

            else:
                wiki_pages = data.from_iter(lambda: iter(()))

            concepts = (legacy_concepts.union(qid_concepts)
                .reduce_by_key('concept_ref', merge_concept)
                .checkpoint(tables/'concepts.jsonl', version=version))
            images = collected_images.checkpoint(tables/'images.jsonl', version=version)

            # 2. 页面关联：概念的lang/page_id → Wiki文档concept_refs。
            # left join保留未匹配页面；多概念对应不自动消歧。其他文档保留采集关联。
            page_refs = (concepts.flat_map(lambda c:c['page_refs'])
                .reduce_by_key(['lang','page_id','mapped_concept_ref'], distinct)
                .checkpoint(tables/'page_refs.jsonl', version=version))
            wiki_documents = (wiki_pages
                .join(page_refs, on=['lang','page_id'], how='left')
                .reduce_by_key('doc_id', merge_document))
            documents = collected_documents.union(wiki_documents)
            documents = documents.checkpoint(tables/'documents.jsonl', version=version)

            # 3. SelectConcept：概念行 → 增加selected/selection_reason，再filter。
            # 与读取端使用同一采样规则；下游业务算子不接收入口概念ID。
            concept_selection = (concepts.map(SelectConcept(ids, sample_rate, seed))
                .checkpoint(tables/'concepts_selected.jsonl', version=version))
            selected_concepts = concept_selection.filter(lambda c:c['selected'])
            selected_keys = selected_concepts.select_columns(['concept_ref'])
            if ids is not None:
                (data.from_iter(lambda:({'concept_ref':ref} for ref in ids))
                    .join(concepts.select_columns(['concept_ref']), on='concept_ref', how='anti')
                    .checkpoint(tables/'missing_concepts.jsonl', version=version))

            # 4. MaterialLinks：文档/图片行 → concept_ref与doc_id/image_id关联键。
            # 分别semi join入选概念，再用资料ID筛选原表；共享资料只处理一次。
            all_document_links = documents.flat_map(MaterialLinks('doc_id'))
            all_image_links = images.flat_map(MaterialLinks('image_id'))
            document_links = (all_document_links.join(selected_keys, on='concept_ref', how='semi')
                .checkpoint(tables/'documents_links.jsonl', version=version))
            image_links = (all_image_links.join(selected_keys, on='concept_ref', how='semi')
                .checkpoint(tables/'images_links.jsonl', version=version))
            selected_documents = (documents.join(document_links.select_columns(['doc_id'])
                .reduce_by_key('doc_id', distinct), on='doc_id', how='semi')
                .checkpoint(tables/'documents_selected.jsonl', version=version))
            selected_images = (images.join(image_links.select_columns(['image_id'])
                .reduce_by_key('image_id', distinct), on='image_id', how='semi')
                .checkpoint(tables/'images_selected.jsonl', version=version))
            # 未关联任何已读概念的资料另存；不是把未入选概念的资料判为无关。
            if global_audit:
                for objects, links, key, name in [
                    (documents, all_document_links, 'doc_id', 'documents'),
                    (images, all_image_links, 'image_id', 'images')]:
                    associated = (links.join(concepts.select_columns(['concept_ref']), on='concept_ref', how='semi')
                        .select_columns([key]).reduce_by_key(key, distinct))
                    objects.join(associated, on=key, how='anti').checkpoint(tables/f'{name}_unmatched.jsonl', version=version)


            # 5. ReadDocument → CleanDocument → FilterDocumentBlocks：读原文、解析正文、过滤与修复。
            # 原文及排除依据保留；重建clean_text和块定位后才交给身份/相关性判断。
            # 输入：选中文档path/sections；输出：raw_text → clean_text/块定位/准入状态。
            # 不删除原文，pending带原因保留；map_cached复用同输入同版本结果。
            processed_documents = (selected_documents
                .map_cached(ReadDocument(dataset), cache_dir=run/'cache/read_documents', version=version)
                .map_cached(CleanDocument(), cache_dir=run/'cache/clean_documents', version=version)
                .map_cached(FilterDocumentBlocks(), cache_dir=run/'cache/filter_documents', version=version)
                .checkpoint(tables/'documents_processed.jsonl', version=version))
            # 6. CheckImage：图片独立扩列；路径/哈希 → byte_status/byte_details。
            # 文件可用性检查不是图片语义核验。
            processed_images = (selected_images
                .map_cached(CheckImage(dataset), cache_dir=run/'cache/check_images', version=version)
                .checkpoint(tables/'images_processed.jsonl', version=version))

            # 可选：读取预标注JSONL冻结快照，按图片SHA关联；不把机器标签当人工核验。
            if config.get('image_annotations_file'):
                annotations = (data.read_records(config['image_annotations_file'])
                    .map(lambda r: {'sha256':r['value']['sha256'],'preannotation':r['value']} if r['error'] is None else (_ for _ in ()).throw(ValueError('invalid annotation row')))
                    .checkpoint(tables/'image_annotations.jsonl',version=version))
                processed_images = (processed_images.join(annotations,on='sha256',how='left')
                    .checkpoint(tables/'images_annotated.jsonl',version=version))

            # 7. CountMaterial：关联键join处理状态后，按概念归约计数。
            # 输出：每概念文档/图片总数、可读/字节通过数；零资料概念left join保留。
            document_counts = (document_links
                .join(processed_documents.select_columns(['doc_id','read_status']), on='doc_id')
                .reduce_by_key('concept_ref', CountMaterial('document_count','read_status','readable_documents')))
            image_counts = (image_links
                .join(processed_images.select_columns(['image_id','byte_status']), on='image_id')
                .reduce_by_key('concept_ref', CountMaterial('image_count','byte_status','verified_images')))
            concepts_ready = (selected_concepts
                .join(document_counts, on='concept_ref', how='left')
                .join(image_counts, on='concept_ref', how='left')
                .map(fill_material_counts)
                .checkpoint(tables/'concepts_ready.jsonl', version=version))

            # 8. NestMaterial + group_batches：到这里才按概念汇集完整文档/图片。
            # 输出knowledge_inputs：每行一个概念材料批次，含materials及批次索引。
            # 这只是执行分批，尚未完成跨批语义整合。
            concept_documents = document_links.join(
                processed_documents.map(NestMaterial('doc_id','documents')), on='doc_id')
            concept_images = image_links.join(
                processed_images.map(NestMaterial('image_id','images')), on='image_id')
            material_batches = concept_documents.union(concept_images).group_batches(
                'concept_ref', max_rows=group_size, output='materials')
            batches = (concepts_ready.join(material_batches, on='concept_ref', how='left')
                .checkpoint(tables/'knowledge_inputs.jsonl', version=version))
            if through == 'gather': return batches

        # 9. ResolveIdentity：概念与清洗资料 → 身份/逐材料依据/接受拒绝/未查看范围。
        # model_input仅转换联合输入格式；身份歧义blocked，图片此时仅看元数据。
        knowledge_run = run/'knowledge'
        pack, prompt_text = knowledge_prompt_pack(config)
        options = prompt_execution_options(run, config)
        save_prompt_config(run, prompt_text, options)
        prompt_data = local_data(prompt_packs={'knowledge.yaml':pack},
                                 max_prompt_requests=config['max_calls'], prompt_options=options)
        # 同一材料Dataset进入原生提示词执行上下文；请求预算和持久账本跨阶段共享。
        if reuse_filter_inputs is not None:
            identified=prompt_data.read_json(str(tables/'knowledge_identity.jsonl'))
        elif reuse_materials is None:
            batches = prompt_data.read_json(str(tables/'knowledge_inputs.jsonl'))
            identified = (batches.map(model_input)
                # 准备：保留源资料，生成identity_prompt；不调用模型。
                .map_cached(PrepareIdentity(knowledge_run, config),
                            cache_dir=knowledge_run/'cache/PrepareIdentity', version=version)
                # 调用：demiflow负责异步HTTP、完整请求响应、预算和精确回放。
                .map_prompt_async('identity', config='knowledge.yaml', inputs={'payload':'identity_prompt'},
                                  output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                                  when=lambda r: not r.get('blocked') and 'identity_prompt' in r,
                                  concurrency=1, queue_depth=1)
                # 校验：结果与原文/材料对应检查；错误保留为blocked，不丢行。
                .map_cached(ApplyIdentity(knowledge_run, config),
                            cache_dir=knowledge_run/'cache/ApplyIdentity', version=version)
                .checkpoint(tables/'knowledge_identity.jsonl', version=version))
            if through == 'identity': return identified

        # 10M. 按概念筛选 → 联合提取 → 一次review并写最终文章。
        # 分批按实际输入容量；所有入选材料进入后续路由。
        if config.get('text_mode') == 'multimodal':
            if reuse_materials is not None:
                related=prompt_data.read_json(str(tables/'related_materials.jsonl'))
            else:
                if reuse_filter_inputs is not None:
                    blocks=prompt_data.read_json(str(tables/'multimodal_materials.jsonl'))
                else:
                    blocks = (identified.map(BuildSourceBlocks(config.get('block_unit_chars',1800), body_only=True))
                        .map(SelectAvailableImages()).checkpoint(tables/'multimodal_materials.jsonl', version=version))
                text_requests = blocks.flat_map(BatchSourceBlocks(config.get('block_batch_chars',8000))).map(PrepareSelectionScope(config['image_identity_definitions']))
                text_decisions = (text_requests.map_prompt_async('select_blocks', config='knowledge.yaml',
                    inputs={'payload':'block_prompt'}, output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                    .map_cached(ApplyBlockSelection(relevance_only=True),cache_dir=knowledge_run/'cache/text_relevance',version=version)
                    .checkpoint(tables/'text_relevance.jsonl',version=version)
                    .reduce_by_key('case_id',merge_block_decisions))
                # 10A. Qwen初筛：只给概念身份资料和像素，不给上游图片结论。
                image_requests = blocks.flat_map(BatchImageSelection(config.get('image_batch_size',4),
                    config['image_identity_definitions'], neutral=True)).checkpoint(tables/'image_requests.jsonl',version=version)
                primary_data = image_prompt_data(run, config)
                primary = (primary_data.read_json(str(tables/'image_requests.jsonl'))
                    .map_prompt_async('select_images',config='knowledge.yaml',
                        inputs={'payload':'image_prompt','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                    .map_cached(RecordPrimaryImageSelection(),cache_dir=run/'cache/image_primary',version=version)
                    .checkpoint(tables/'image_primary.jsonl',version=version))
                # 10B. 含入选图才复核；保留整个原批次，Gemma看不到Qwen判断。
                review_rows = primary.map(PrepareImageReview()).checkpoint(tables/'image_review_inputs.jsonl',version=version)
                review_data = image_prompt_data(run, config, review=True)
                review_requests = review_data.read_json(str(tables/'image_review_inputs.jsonl')).filter(lambda r:r['review_required'])
                review_path = tables/'image_review_responses.jsonl'
                # 只在有未完成复核时借用GPU；该阶段落盘后恢复Qwen及预标注。
                with image_review_service(run, config, needed=review_needed(review_requests,review_path,version)):
                    reviewed = (review_requests.map_prompt_async('select_images',config='knowledge.yaml',
                        inputs={'payload':'image_prompt','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',
                        concurrency=config['image_review_concurrency'],queue_depth=config['image_review_concurrency'])
                        .checkpoint(review_path,version=version))
                # 10C. 原排除/待定继续保留；入选图须独立复核keep，否则暂缓。
                image_decisions = (reviewed.union(review_rows.filter(lambda r:not r['review_required']))
                    .map_cached(ApplyConfirmedImageSelection(),cache_dir=run/'cache/image_confirmed',version=version)
                    .checkpoint(tables/'image_relevance.jsonl',version=version)
                    .reduce_by_key('case_id',merge_image_decisions))
                related = (blocks.join(text_decisions,on='case_id',how='left')
                    .join(image_decisions,on='case_id',how='left').map(SelectRelatedMaterials())
                    .checkpoint(tables/'related_materials.jsonl',version=version))
                save_image_filter_policy(run,config)
                if through=='organize':return related
            # 11. 原生图文关联 → 文本embedding分组 → 图文embedding补充关联。
            routing_materials = related.map(PrepareRoutingMaterials()).checkpoint(tables/'routing_materials.jsonl',version=version)
            text_embeddings = (routing_materials.flat_map(RawPassageRows())
                .group_batches('embedding_bucket',max_rows=2,output='items')
                .map_cached(EmbedParagraphBatch(config['text_embedding_model']),cache_dir=run/'cache/material_text_embedding',version=version)
                .checkpoint(tables/'material_text_embeddings.jsonl',version=version)
                .flat_map(lambda r:r['items']).reduce_by_key('case_id',
                    lambda acc,r:{'case_id':r['case_id'],'passage_embeddings':{**acc['passage_embeddings'],r['source_id']:r}},initial={'passage_embeddings':{}}))
            image_text_embeddings = (routing_materials
                .map_cached(EncodeImageTextMaterials(config['image_embedding_model']),cache_dir=run/'cache/material_image_embedding',version=version)
                .checkpoint(tables/'material_image_embeddings.jsonl',version=version))
            if reuse_materials is not None:save_image_filter_policy(run,config)
            routed = (routing_materials.join(text_embeddings,on='case_id',how='left')
                .join(image_text_embeddings.select_columns(['case_id','text_windows','image_vectors']),on='case_id',how='left')
                .map(RouteByTokenBudget(ROOT.parent/'models/Qwen3.8-27B', config.get('joint_input_tokens',32768), counter=ArticleTokenBudget(ROOT.parent/'models/Qwen3.8-27B', {**config,'enable_thinking':config.get('joint_thinking',True),'reasoning_effort':config.get('joint_reasoning_effort','low')}, 'joint_paragraphs')))
                .checkpoint(tables/'material_routing.jsonl',version=version))
            # 12. 容量组装，不做文字×图片全组合；图片数量是单次输入目标，非最终保留上限。
            joint_requests = (routed.flat_map(lambda r:r['requests']).map(BuildRoutedJointRequest())
                .checkpoint(tables/'joint_requests.jsonl',version=version))
            # 13. joint_paragraphs：简明原文与真实图片 → 带资料/图片标记的提取稿。
            joint_config={**config,'enable_thinking':config.get('joint_thinking',True),'reasoning_effort':config.get('joint_reasoning_effort','low')}
            joint_options=prompt_execution_options(run,joint_config)
            save_prompt_config(run/'joint_extraction',prompt_text,joint_options)
            joint_data=local_data(prompt_packs={'knowledge.yaml':pack},prompt_options=joint_options,max_prompt_requests=config['max_calls'])
            article_inputs = joint_data.read_json(str(tables/'joint_requests.jsonl')).map(PrepareArticleInput(config['image_identity_definitions'])).checkpoint(tables/'article_inputs.jsonl',version=version)
            extracted = (article_inputs.map_prompt_async('joint_paragraphs',config='knowledge.yaml',
                inputs={'payload':'article_input','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                .map_cached(ApplyArticle(),cache_dir=run/'cache/paragraph_extract',version=version)
                .checkpoint(tables/'paragraph_extract.jsonl',version=version))
            if through=='extract':return extracted
            # 14. 收齐同概念所有提取组，仅携带实际引用的原文及图片；超限明确阻塞。
            draft_groups = extracted.reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'drafts':acc['drafts']+[r]},initial={'drafts':[]})
            review_config={**config,'enable_thinking':config.get('final_review_thinking',True),'reasoning_effort':config.get('final_review_effort','low')}
            review_options=prompt_execution_options(run,review_config)
            save_prompt_config(run/'final_review',prompt_text,review_options)
            review_data=local_data(prompt_packs={'knowledge.yaml':pack},prompt_options=review_options,max_prompt_requests=config['max_calls'])
            review_requests = (draft_groups.map(PrepareFinalReview(config['image_identity_definitions'],
                ArticleTokenBudget(ROOT.parent/'models/Qwen3.8-27B', review_config, 'final_review'),config.get('final_input_tokens',131072)))
                .checkpoint(tables/'final_review_requests.jsonl',version=version))
            # 15. final_review：一次检查并写最终文章；没有另外的语义合并/修复链。
            reviewed = (review_data.read_json(str(tables/'final_review_requests.jsonl')).map_prompt_async('final_review',config='knowledge.yaml',
                inputs={'payload':'article_input','images':'pixel_images'},when=lambda r:not r['preflight_error'],
                output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                .map_cached(ApplyArticle(final=True),cache_dir=run/'cache/final_review',version=version)
                .checkpoint(tables/'final_review.jsonl',version=version))
            if through in {'consolidate','fidelity','evidence'}:return reviewed
            # 16. 程序按真实编号附回来源/图片；失败不发布提取稿，零知识概念也保留状态。
            material_groups = (related.map(lambda r:{'concept':r['identity']['target_label'],'material':r})
                .reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'materials':acc['materials']+[r['material']]},initial={'materials':[]}))
            return (material_groups.join(reviewed.map(lambda r:{'concept':r['concept'],'review':r}),on='concept',how='left')
                .map(PublishArticle()).checkpoint(run/'knowledge_base.jsonl',version=version))

        raise ValueError('Formal notebook supports text_mode=multimodal only')


# 唯一执行入口；run/dataset/sources是显式参数，没有f/StreamFlow。
if MODE == 'execute':
    final_dataset = await asyncio.to_thread(
        run_pipeline, RUN, DATASET, ids=IDS, sample_rate=SAMPLE_RATE,
        seed=SEED, max_records_per_source=MAX_RECORDS, group_size=GROUP_SIZE,
        through=THROUGH, model_config=MODEL_CONFIG, source_scope=SOURCE_SCOPE, reuse_materials=REUSE_MATERIALS)
else:
    final_dataset = None  # 只读模式由下方最终结果查看器读取FINAL_FILE
print(f'模式：{MODE}；主链终点：{THROUGH}；结果目录：{RUN}')
if MODE == 'execute' and THROUGH == 'export': FINAL_FILE = RUN/'knowledge_base.jsonl'


## 3．当前 RUN 最终结果

只读取上面配置的 RUN；尚未完成时显示缺失提示，不回退到任何历史实验。仅展示标题、正文与入选图片、参考来源。

In [ ]:
import importlib
from curation.v4 import current_results
importlib.reload(current_results)  # 仅刷新展示模块，不热加载业务算子
from curation.v4.current_results import show_current_results
show_current_results(RUN if MODE == 'execute' else SAVED_RUN, **VIEW)